# Stage 18E: build fold-safe ranked-retrieval package

Stage 18Dで昇格した5個のfold-safe rankerとInternet-OFF test inferenceを1つのzipへまとめます。学習済みpackageの作成だけを行い、submissionは作りません。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
REPOSITORY_URL = 'https://github.com/Okada-N13/rogii.git'
repo_dir = Path('/content/ROGII')
drive_root = Path('/content/drive/MyDrive/kaggle/rogii')
artifact_dir = drive_root / 'artifacts'
if not (repo_dir / '.git').is_dir():
    subprocess.run(['git', 'clone', REPOSITORY_URL, str(repo_dir)], check=True)
else:
    subprocess.run(['git', '-C', str(repo_dir), 'pull', '--ff-only', 'origin', 'main'], check=True)
if shutil.which('uv') is None:
    subprocess.run(['bash', '-lc', 'curl -LsSf https://astral.sh/uv/install.sh | sh'], check=True)
os.environ['PATH'] = '/root/.local/bin:' + os.environ['PATH']
subprocess.run(['uv', 'sync', '--frozen'], cwd=repo_dir, check=True)
artifact_dir.mkdir(parents=True, exist_ok=True)
def run_checked(command):
    result = subprocess.run(command, cwd=repo_dir, text=True, capture_output=True)
    if result.stdout:
        print(result.stdout, flush=True)
    if result.returncode != 0:
        print(result.stderr, flush=True)
        raise RuntimeError(f'Command failed with exit code {result.returncode}: {command}')
    return result
subprocess.run(['git', '-C', str(repo_dir), 'rev-parse', '--short', 'HEAD'], check=True)

## package構築

Stage 18Dのcandidate checkpointから、評価foldをtarget/donor両roleで除外した5モデルを再学習します。test wellと同じIDのtrain wellをdonorにする経路は推論コードで明示的に禁止します。

In [ ]:
stage16b_run = artifact_dir / 'stage16b_testlike_validation_full_v003'
stage18d_run = artifact_dir / 'stage18d_learned_donor_ranker_full_v001'
data_dir = drive_root / 'data'
assert (stage16b_run / 'well_assignments.parquet').is_file()
assert (stage18d_run / 'donor_training_rows.parquet').is_file()
assert (data_dir / 'train').is_dir()
assert json.loads((stage18d_run / 'summary.json').read_text())['promoted_to_full_ranker_training']
RUN_ID = 'stage18e_ranked_retrieval_package_v002'
run_dir = artifact_dir / RUN_ID
if not (run_dir / 'summary.json').is_file():
    run_checked([
        'uv', 'run', 'rogii-stage18-package',
        '--stage16b-run', str(stage16b_run), '--stage18d-run', str(stage18d_run),
        '--data-dir', str(data_dir),
        '--artifact-dir', str(artifact_dir), '--run-id', RUN_ID,
    ])
else:
    print('Reusing completed package:', run_dir)
summary = json.loads((run_dir / 'summary.json').read_text())
summary

最後のsummaryを共有してください。生成された`stage18e_ranked_retrieval_package.zip`を次にKaggle Datasetへアップロードします。Dataset名は`rogii-stage18e-ranked-retrieval-package`を推奨します。